In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

# Load data
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [3]:
# Fill missing values with median for numerical features
for col in train_data.select_dtypes(include=[np.number]).columns:
    train_data[col].fillna(train_data[col].median(), inplace=True)

In [4]:
# Fill missing values with median for numerical features
for col in test_data.select_dtypes(include=[np.number]).columns:
    test_data[col].fillna(test_data[col].median(), inplace=True)

In [5]:
# Fill missing values with mode for categorical features
for col in train_data.select_dtypes(include='object').columns:
    train_data[col].fillna(train_data[col].mode()[0], inplace=True)
    test_data[col].fillna(test_data[col].mode()[0], inplace=True)

for col in train_data.columns:
    if train_data[col].dtype == 'object':
        le = LabelEncoder()
        le.fit(list(train_data[col].values) + list(test_data[col].values))
        train_data[col] = le.transform(list(train_data[col].values))
        test_data[col] = le.transform(list(test_data[col].values))

In [6]:
# Adding new features
train_data['TotalSF'] = train_data['TotalBsmtSF'] + train_data['1stFlrSF'] + train_data['2ndFlrSF']
train_data['TotalBathrooms'] = train_data['FullBath'] + (0.5 * train_data['HalfBath']) + train_data['BsmtFullBath'] + (0.5 * train_data['BsmtHalfBath'])
train_data['TotalPorchSF'] = train_data['OpenPorchSF'] + train_data['EnclosedPorch'] + train_data['3SsnPorch'] + train_data['ScreenPorch']

test_data['TotalSF'] = test_data['TotalBsmtSF'] + test_data['1stFlrSF'] + test_data['2ndFlrSF']
test_data['TotalBathrooms'] = test_data['FullBath'] + (0.5 * test_data['HalfBath']) + test_data['BsmtFullBath'] + (0.5 * test_data['BsmtHalfBath'])
test_data['TotalPorchSF'] = test_data['OpenPorchSF'] + test_data['EnclosedPorch'] + test_data['3SsnPorch'] + test_data['ScreenPorch']

# Encoding categorical features
categorical_cols = [col for col in train_data.columns if train_data[col].dtype == 'object']
for col in categorical_cols:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col])
    test_data[col] = le.transform(test_data[col])

In [7]:
# Split data into X and y
X = train_data.drop(['Id', 'SalePrice'], axis=1)
y = train_data['SalePrice']

In [8]:
# Hyperparameter tuning
params = {
    'boosting_type': ['gbdt'],
    'objective': ['regression'],
    'learning_rate': [0.05, 0.1, 0.15],
    'num_leaves': [30, 50, 70],
    'max_depth': [5, 7, 10],
    'feature_fraction': [0.7, 0.8, 0.9],
    'bagging_fraction': [0.7, 0.8, 0.9],
    'bagging_freq': [5, 10, 15]
}

In [9]:
# Model training
gbm = lgb.LGBMRegressor()
grid_search = GridSearchCV(gbm, params, cv=5, verbose=1, n_jobs=-1)
grid_search.fit(X, y)

print('Best parameters found by grid search:')
print(grid_search.best_params_)

Fitting 5 folds for each of 729 candidates, totalling 3645 fits
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[Light

In [10]:
# Model training
model = lgb.LGBMRegressor(**grid_search.best_params_, n_estimators=100)
model.fit(X, y)

LGBMRegressor(bagging_fraction=0.7, bagging_freq=15, feature_fraction=0.9,
              learning_rate=0.05, max_depth=7, num_leaves=50,
              objective='regression')

In [11]:
# Model evaluation
y_pred = model.predict(X)
mse = np.mean((y - y_pred)**2)
rmse = np.sqrt(mse)
print(f'RMSE: {rmse}')

RMSE: 19473.397292149086


In [12]:
# Model predict
X_test = test_data.drop(['Id'], axis=1)
y_test_pred = model.predict(X_test)

In [13]:
# Create submission file
submission = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': y_test_pred})
submission.to_csv('submission.csv', index=False)